Cell 1 — Title (markdown)


# Adaptive Rolling Dynamic RFM — Online Retail II (Research-Validated v6.1)

**Mục tiêu v6.1:** đưa notebook lên mức thực nghiệm có thể audit, không tối ưu chỉ để làm Adaptive thắng. Đây là bản nâng cấp phương pháp; không bảo đảm Adaptive sẽ có metric cao hơn baseline.

Các thay đổi chính:
1. **Cleaning audit + hash-validated cache:** báo cáo số dòng bị loại ở từng bước và không tái sử dụng cache sai dataset.
2. **Matched fixed baselines:** bổ sung Fixed 14W, 16W và 26W (98, 112, 182 ngày) để kiểm tra lợi ích cá nhân hóa so với độ dài cửa sổ tương đương.
3. **Population-aware clustering metrics:** báo cáo Silhouette/DBI/CH trên Active và All riêng; không xếp Active-only chung với All-population.
4. **Stability split:** báo cáo ổn định cluster trên khách Active ở cả hai snapshot và trạng thái Active/Inactive riêng.
5. **No composite rank-sum:** tách Quality, Stability và Predictive Utility; không cộng thứ hạng metric khác bản chất.
6. **Non-circular block bootstrap:** giữ nguyên vị trí thời gian và NaN; CI bị từ chối khi không đủ block liên tục.
7. **Churn exploratory only:** không đưa Churn vào bảng xếp hạng chính hoặc kết luận tổng hợp.
8. **Ablation minh bạch:** cluster feature chỉ là thí nghiệm phụ; `-1` của Two-stage được ghi nhận là encoding trạng thái inactive, không phải thông tin mới độc lập.
9. **Integrity check:** chỉ liệt kê output thực sự được tạo; không tuyên bố Walk-forward nếu không chạy.

> Chạy **Runtime → Run all** từ đầu. Notebook sẽ yêu cầu upload ZIP hoặc XLSX của Online Retail II. Mọi số liệu phải được tạo lại trong runtime hiện tại; không có số liệu thực nghiệm giả định được điền sẵn.

### Phạm vi diễn giải
- Primary segmentation quality: đánh giá trên **khách Active-in-window** để tránh Silhouette bị chi phối bởi nhóm F=0.
- `silhouette_all`: chỉ là metric phụ mô tả toàn bộ population.
- Prediction AUC/AP: **cross-sectional future-label discrimination** trong từng snapshot, chưa phải temporal deployment test.
- Churn: operational inactivity với horizon 90 ngày, chỉ báo cáo exploratory.
- Clustering dùng reference model fit ở cuối train và giữ cố định trong holdout; đây chưa phải full dynamic reclustering.

## Cell 2 — Setup & imports

In [ ]:
%pip -q install "openpyxl>=3.1,<4" "scikit-learn>=1.4,<2" "scipy>=1.11,<2" "pyarrow>=14,<25" "matplotlib>=3.7,<4"

from pathlib import Path
import pandas as pd
import numpy as np
import logging, sys, json, zipfile, shutil, warnings, time, platform, hashlib, importlib.metadata
from dataclasses import dataclass, asdict
from typing import List, Optional, Sequence
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, roc_auc_score, average_precision_score,
)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import wilcoxon
from itertools import combinations

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
LOG = logging.getLogger("rfm")

RFM_COLS = ["Recency", "Frequency", "Monetary"]
RFM_RATE_COLS = ["Recency", "FrequencyRate30", "MonetaryRate30"]
SEED = 42
np.random.seed(SEED)

print("Imports OK. Dependency constraints installed; exact runtime versions are recorded in the manifest.")

## Cell 3 — Reproducibility manifest

In [ ]:
import sklearn, scipy, matplotlib

REPRO = {
    "notebook_version": "v6.1",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seed_global": SEED,
    "runtime_note": "Version constraints are installed; exact resolved versions are recorded after runtime initialization.",
    "hyperparams_static": {
        "N_CLUSTERS": 5,
        "FUTURE_HORIZON": 28,
        "CHURN_HORIZON": 90,
        "CHURN_MIN_HISTORY": 90,
        "N_BOOT": 2000,
        "SNAPSHOT_STEP": 14,
        "TRAIN_RATIO": 0.70,
        "PRIMARY_METRIC_POPULATION": "active_in_window",
    },
}
print("Reproducibility manifest initialized.")
print(json.dumps(REPRO, indent=2))

## Cell 4 — Upload Online Retail II (ZIP hoặc XLSX)

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

uploaded_name = next(iter(uploaded))
uploaded_path = Path(uploaded_name)
if uploaded_path.suffix.lower() == ".zip":
    with zipfile.ZipFile(uploaded_path) as zf:
        xlsx_members = [n for n in zf.namelist() if n.lower().endswith((".xlsx", ".xls"))]
        if not xlsx_members:
            raise FileNotFoundError("ZIP does not contain an Excel workbook.")
        # Prefer a workbook whose name contains 'online' or 'retail'; otherwise use the first.
        xlsx_members.sort(key=lambda x: ("online" not in x.lower() and "retail" not in x.lower(), len(x)))
        xlsx = xlsx_members[0]
        zf.extract(xlsx, "/content")
    raw_path = Path("/content") / xlsx
elif uploaded_path.suffix.lower() in {".xlsx", ".xls"}:
    raw_path = uploaded_path.resolve()
else:
    raise ValueError("Please upload an Online Retail II ZIP or Excel file.")

cache_path = Path("/content/online_retail_ii_v61.parquet")
print("Raw:", raw_path, "| Cache:", cache_path)

## Cell 5 — Load, clean, audit population, validate cache by source hash

In [ ]:
def _file_sha256(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def load_and_clean(raw_path: Path, cache_path: Path, rebuild: bool = False):
    raw_hash = _file_sha256(raw_path)
    meta_path = cache_path.with_suffix(".meta.json")
    audit_path = cache_path.with_suffix(".cleaning_audit.csv")
    cleaning_version = "online-retail-clean-v2"

    # Cache is reused only when its source hash AND cleaning version match.
    if cache_path.exists() and meta_path.exists() and audit_path.exists() and not rebuild:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("source_sha256") == raw_hash and meta.get("cleaning_version") == cleaning_version:
            LOG.info("Loading hash-validated cache: %s", cache_path)
            cached = pd.read_parquet(cache_path)
            audit = pd.read_csv(audit_path) if audit_path.exists() else pd.DataFrame()
            return cached, audit, raw_hash
        LOG.warning("Cache metadata mismatch; rebuilding from uploaded source.")

    LOG.info("Reading workbook; this is usually the slowest first-run step.")
    sheets = pd.read_excel(raw_path, sheet_name=None)
    parts = []
    for sheet_name, part in sheets.items():
        if part is None or part.empty:
            continue
        part = part.copy()
        part.columns = [" ".join(str(c).replace("\\ufeff", "").strip().split()) for c in part.columns]
        parts.append(part)
    if not parts:
        raise ValueError("No non-empty worksheets found in workbook.")
    raw = pd.concat(parts, ignore_index=True)
    raw_rows = len(raw)

    rename = {}
    for col in raw.columns:
        key = str(col).lower().replace(" ", "").replace("_", "")
        if key in {"invoice", "invoiceno"}: rename[col] = "Invoice"
        elif key in {"quantity", "qty"}: rename[col] = "Quantity"
        elif key in {"invoicedate", "date"}: rename[col] = "InvoiceDate"
        elif key in {"price", "unitprice"}: rename[col] = "Price"
        elif key in {"customerid", "customer"}: rename[col] = "CustomerID"
    raw = raw.rename(columns=rename)
    required = ["Invoice", "Quantity", "InvoiceDate", "Price", "CustomerID"]
    missing_cols = [c for c in required if c not in raw.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df0 = raw.copy()
    df0["Invoice"] = df0["Invoice"].astype("string").str.strip()
    for col in ["Quantity", "Price", "CustomerID"]:
        df0[col] = pd.to_numeric(df0[col], errors="coerce")
    df0["InvoiceDate"] = pd.to_datetime(df0["InvoiceDate"], errors="coerce")

    audit_rows = []
    def audit_step(name, before, removed, after, note=""):
        audit_rows.append({"step": name, "rows_before": int(before),
                           "rows_removed": int(removed), "rows_remaining": int(after),
                           "note": note})

    before = len(df0)
    missing_customer_id = int(df0["CustomerID"].isna().sum())
    missing_any = df0[required].isna().any(axis=1)
    df1 = df0.loc[~missing_any].copy()
    audit_step("drop_missing_required", before, before - len(df1), len(df1),
               f"Missing CustomerID in raw rows: {missing_customer_id:,}")

    before = len(df1)
    cancelled = df1["Invoice"].str.upper().str.startswith("C", na=False)
    df2 = df1.loc[~cancelled].copy()
    audit_step("remove_cancelled_invoices", before, before - len(df2), len(df2))

    before = len(df2)
    valid_positive = (df2["Quantity"] > 0) & (df2["Price"] > 0)
    df3 = df2.loc[valid_positive].copy()
    audit_step("require_positive_quantity_price", before, before - len(df3), len(df3))

    df3["CustomerID"] = df3["CustomerID"].astype("int64")
    df3["Amount"] = df3["Quantity"].astype(float) * df3["Price"].astype(float)
    df3["InvoiceDate"] = df3["InvoiceDate"].dt.normalize()
    df3 = df3.sort_values("InvoiceDate").reset_index(drop=True)
    audit_step("final_cleaned", raw_rows, raw_rows - len(df3), len(df3),
               "Final count is cumulative removal from raw rows.")

    audit = pd.DataFrame(audit_rows)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    df3.to_parquet(cache_path, index=False, compression="zstd")
    audit.to_csv(audit_path, index=False)
    meta = {
        "source_file": str(raw_path.name),
        "source_sha256": raw_hash,
        "cleaning_version": cleaning_version,
        "raw_rows": int(raw_rows),
        "cleaned_rows": int(len(df3)),
        "customers": int(df3["CustomerID"].nunique()),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return df3, audit, raw_hash


df, cleaning_audit, dataset_hash = load_and_clean(raw_path, cache_path, rebuild=False)
REPRO.update({
    "dataset_sha256": dataset_hash,
    "dataset_rows_cleaned": int(len(df)),
    "dataset_customers": int(df["CustomerID"].nunique()),
    "dataset_date_min": str(df["InvoiceDate"].min().date()),
    "dataset_date_max": str(df["InvoiceDate"].max().date()),
    "cleaning_audit": cleaning_audit.to_dict(orient="records"),
})
print(f"Cleaned rows: {len(df):,} | Customers: {df.CustomerID.nunique():,}")
print(f"Date range: {df.InvoiceDate.min().date()} → {df.InvoiceDate.max().date()}")
print(f"Source SHA256: {dataset_hash}")
print("\nDATA-CLEANING AUDIT")
display(cleaning_audit)
display(df.head(3))

Cell 6 — Windowed RFM (với assertions + fix dt.days)


In [ ]:
@dataclass(frozen=True)   # ── V6 CHANGE: frozen để tránh mutate
class RFMParams:
    window_days: int = 91
    adaptive_multiplier: float = 2.0
    adaptive_min: int = 28
    adaptive_max: int = 91
    adaptive_default: int = 56
    adaptive_lookback: int = 365
    adaptive_method: str = "hierarchical"
    adaptive_shrinkage_k: float = 3.0
    adaptive_min_cohort_n: int = 20
    feature_mode: str = "rate"
    active_only: bool = False


def precompute_daily(data: pd.DataFrame) -> pd.DataFrame:
    required = {"CustomerID", "InvoiceDate", "Invoice", "Amount"}
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(f"Missing columns for daily aggregation: {sorted(missing)}")
    daily = (
        data.assign(Date=pd.to_datetime(data["InvoiceDate"]).dt.normalize())
        .groupby(["CustomerID", "Date"], as_index=False)
        .agg(n_inv=("Invoice", "nunique"), amount=("Amount", "sum"))
        .sort_values(["CustomerID", "Date"])
        .reset_index(drop=True)
    )
    return daily


def _adaptive_windows(hist_daily, customer_index, snapshot, params):
    lookback_start = snapshot - pd.Timedelta(days=params.adaptive_lookback)
    recent = hist_daily.loc[
        hist_daily["Date"] >= lookback_start, ["CustomerID", "Date"]
    ].drop_duplicates().sort_values(["CustomerID", "Date"])

    n_days = recent.groupby("CustomerID")["Date"].nunique().reindex(customer_index).fillna(0)
    gaps = recent.copy()
    gaps["GapDays"] = gaps.groupby("CustomerID")["Date"].diff().dt.days

    gap_stats = gaps.groupby("CustomerID")["GapDays"].agg(
        MedianGapDays="median", NGap="count"
    ).reindex(customer_index)
    gap_stats["NGap"] = gap_stats["NGap"].fillna(0).astype(int)
    gap_stats["NDays"] = n_days.astype(int)

    observed = gap_stats["MedianGapDays"].dropna()
    global_prior = float(observed.median()) if len(observed) else np.nan
    if not np.isfinite(global_prior) or global_prior <= 0:
        global_prior = max(1.0, params.adaptive_default / max(params.adaptive_multiplier, 1e-9))

    if params.adaptive_method == "individual":
        effective_gap = gap_stats["MedianGapDays"].fillna(
            params.adaptive_default / max(params.adaptive_multiplier, 1e-9)
        )
        prior_gap = pd.Series(global_prior, index=customer_index)
        source = pd.Series(
            np.where(gap_stats["NGap"].to_numpy() > 0, "individual", "global_prior"),
            index=customer_index, dtype="object",
        )
        personal_weight = (gap_stats["NGap"] > 0).astype(float)
    elif params.adaptive_method == "hierarchical":
        bands = pd.cut(
            gap_stats["NDays"],
            bins=[-1, 0, 1, 3, 7, 14, np.inf],
            labels=["0_days", "1_day", "2_3_days", "4_7_days", "8_14_days", "15plus_days"],
            include_lowest=True,
        ).astype("object")
        gap_stats["Cohort"] = bands

        cohort_stats = (
            gap_stats.dropna(subset=["MedianGapDays"])
            .groupby("Cohort", observed=False)["MedianGapDays"]
            .agg(["median", "count"])
        )
        cohort_priors = {}
        for cohort, row in cohort_stats.iterrows():
            if pd.notna(row["median"]) and row["count"] >= params.adaptive_min_cohort_n:
                cohort_priors[cohort] = float(row["median"])

        prior_gap = bands.map(cohort_priors).astype(float)
        prior_source = pd.Series(
            np.where(prior_gap.notna(), "cohort_prior", "global_prior"),
            index=customer_index, dtype="object",
        )
        prior_gap = prior_gap.fillna(global_prior)

        n_gap = gap_stats["NGap"].astype(float)
        k = max(float(params.adaptive_shrinkage_k), 0.0)
        weight = n_gap / (n_gap + k) if k > 0 else (n_gap > 0).astype(float)
        personal_weight = pd.Series(weight, index=customer_index, dtype=float)
        personal_gap = gap_stats["MedianGapDays"].fillna(prior_gap)
        effective_gap = weight * personal_gap + (1.0 - weight) * prior_gap

        source = prior_source.copy()
        enough_personal = gap_stats["NGap"] >= 4
        source.loc[enough_personal] = "individual_dominant"
        source.loc[(gap_stats["NGap"] > 0) & ~enough_personal] = "shrunk_personal_cohort"
    else:
        raise ValueError("adaptive_method must be 'individual' or 'hierarchical'.")

    effective_gap = pd.Series(effective_gap, index=customer_index).clip(lower=1.0)
    raw_window = effective_gap * params.adaptive_multiplier
    window = raw_window.clip(
        lower=params.adaptive_min, upper=params.adaptive_max,
    ).round().astype("int64")

    diagnostics = pd.DataFrame(index=customer_index)
    diagnostics["Window"] = window
    diagnostics["AdaptiveFallback"] = gap_stats["NGap"].eq(0)
    diagnostics["MedianGapDays"] = gap_stats["MedianGapDays"]
    diagnostics["NGap"] = gap_stats["NGap"]
    diagnostics["CohortPriorGapDays"] = prior_gap
    diagnostics["EffectiveGapDays"] = effective_gap
    diagnostics["PersonalWeight"] = pd.Series(personal_weight, index=customer_index).fillna(0.0)
    diagnostics["CohortWeight"] = 1.0 - diagnostics["PersonalWeight"]
    diagnostics["AdaptiveSource"] = source
    return diagnostics


def compute_windowed_rfm(data, snapshot, params, daily_agg=None):
    snapshot = pd.Timestamp(snapshot).normalize()
    if daily_agg is None:
        daily_agg = precompute_daily(data)

    hist_daily = daily_agg.loc[daily_agg["Date"] < snapshot].copy()
    out_cols = [
        "CustomerID", *RFM_COLS, "Window", "FrequencyRate30", "MonetaryRate30",
        "AdaptiveFallback", "MedianGapDays", "NGap", "CohortPriorGapDays",
        "EffectiveGapDays", "PersonalWeight", "CohortWeight", "AdaptiveSource",
    ]
    if hist_daily.empty:
        return pd.DataFrame(columns=out_cols)

    last = hist_daily.groupby("CustomerID")["Date"].max()
    # ── V6 CHANGE: dt.days đã là int64, bỏ .astype("int64") thừa ──
    recency = (snapshot - last).dt.days.rename("Recency")
    customer_index = recency.index

    if params.window_days > 0:
        window = pd.Series(params.window_days, index=customer_index,
                           name="Window", dtype="int64")
        adaptive_diag = pd.DataFrame(index=customer_index)
        adaptive_diag["AdaptiveFallback"] = False
        adaptive_diag["MedianGapDays"] = np.nan
        adaptive_diag["NGap"] = 0
        adaptive_diag["CohortPriorGapDays"] = np.nan
        adaptive_diag["EffectiveGapDays"] = np.nan
        adaptive_diag["PersonalWeight"] = np.nan
        adaptive_diag["CohortWeight"] = np.nan
        adaptive_diag["AdaptiveSource"] = "fixed"
    else:
        adaptive_diag = _adaptive_windows(hist_daily, customer_index, snapshot, params)
        window = adaptive_diag["Window"]

    in_window = hist_daily.copy()
    in_window["Window"] = in_window["CustomerID"].map(window)
    days_ago = (snapshot - in_window["Date"]).dt.days
    in_window = in_window.loc[(days_ago > 0) & (days_ago <= in_window["Window"])]

    freq = in_window.groupby("CustomerID")["n_inv"].sum().rename("Frequency")
    monetary = in_window.groupby("CustomerID")["amount"].sum().rename("Monetary")

    result = pd.concat(
        [recency, freq, monetary, window.rename("Window"),
         adaptive_diag.drop(columns=["Window"], errors="ignore")],
        axis=1,
    )
    result[["Frequency", "Monetary"]] = result[["Frequency", "Monetary"]].fillna(0)
    result["Frequency"] = result["Frequency"].astype("int64")
    result["Monetary"] = result["Monetary"].astype(float)
    result["Window"] = result["Window"].astype("int64")
    result["AdaptiveFallback"] = result["AdaptiveFallback"].fillna(False).astype(bool)
    result["FrequencyRate30"] = result["Frequency"] * 30.0 / result["Window"]
    result["MonetaryRate30"] = result["Monetary"] * 30.0 / result["Window"]

    return result.reset_index()


# ── V6 CHANGE: assertion helper ──
def assert_rfm_invariants(rfm: pd.DataFrame, params: RFMParams):
    if rfm.empty:
        return
    assert (rfm["Frequency"] >= 0).all(), "Negative Frequency"
    assert (rfm["Monetary"] >= 0).all(), "Negative Monetary"
    assert (rfm["Recency"] >= 0).all(), "Negative Recency"
    assert (rfm["Window"] > 0).all(), "Non-positive Window"
    if params.window_days == 0:
        assert rfm["Window"].between(params.adaptive_min, params.adaptive_max).all(), \
            "Adaptive window outside [min, max]"
    else:
        assert (rfm["Window"] == params.window_days).all(), \
            "Fixed window mismatch"


# ── Sanity check (float tolerance fix) ──
snap_test = pd.Timestamp(df["InvoiceDate"].max()).normalize() + pd.Timedelta(days=1)
daily_all_check = precompute_daily(df)
fixed_check = RFMParams(window_days=91)
rfm_check = compute_windowed_rfm(df, snap_test, fixed_check, daily_agg=daily_all_check)
assert_rfm_invariants(rfm_check, fixed_check)

raw_hist = df.loc[df["InvoiceDate"] < snap_test]
raw_last = raw_hist.groupby("CustomerID")["InvoiceDate"].max()
raw_recency = (snap_test - raw_last).dt.days
raw_window = raw_hist.loc[raw_hist["InvoiceDate"] >= snap_test - pd.Timedelta(days=91)]
raw_freq = raw_window.groupby("CustomerID")["Invoice"].nunique()
raw_monetary = raw_window.groupby("CustomerID")["Amount"].sum()
check = rfm_check.set_index("CustomerID")

assert np.allclose(check.loc[raw_recency.index, "Recency"].to_numpy(),
                   raw_recency.to_numpy(), rtol=0, atol=0), "Recency mismatch"
assert np.allclose(check.loc[raw_recency.index, "Frequency"].to_numpy(),
                   raw_freq.reindex(raw_recency.index).fillna(0).astype(int).to_numpy(),
                   rtol=0, atol=0), "Frequency mismatch"
assert np.allclose(check.loc[raw_recency.index, "Monetary"].to_numpy(),
                   raw_monetary.reindex(raw_recency.index).fillna(0).to_numpy(),
                   rtol=1e-9, atol=1e-6), "Monetary mismatch"
print(f"PASS: Raw vs daily RFM matched for {len(rfm_check):,} customers.")

## Cell 7 — Clustering model (fit preprocessing on train only)

In [ ]:
@dataclass(frozen=True)   # ── V6 CHANGE: frozen
class ClusteringConfig:
    n_clusters: int = 5
    random_state: int = 42
    batch_size: int = 2048
    n_init: int = 5
    max_iter: int = 180
    metric_sample: int = 2000
    winsor_quantile: float = 0.995


def _feature_matrix(rfm, feature_cols, clip_values=None):
    missing = set(feature_cols).difference(rfm.columns)
    if missing:
        raise ValueError(f"RFM feature columns missing: {sorted(missing)}")
    X = rfm[list(feature_cols)].to_numpy(dtype=np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X = np.maximum(X, 0.0)
    if clip_values is not None:
        X = np.minimum(X, np.asarray(clip_values, dtype=float))
    return np.log1p(X).astype(np.float32)


def fit_reference(rfm, cfg, feature_cols=None, active_only=False):
    feature_cols = list(feature_cols or RFM_RATE_COLS)
    fit_rfm = rfm.loc[rfm["Frequency"] > 0].copy() if active_only else rfm.copy()
    if len(fit_rfm) < max(cfg.n_clusters * 5, 50):
        raise ValueError(f"Too few customers to fit {cfg.n_clusters} clusters: {len(fit_rfm)}")

    raw = fit_rfm[feature_cols].to_numpy(dtype=float)
    raw = np.maximum(np.nan_to_num(raw, nan=0.0, posinf=0.0), 0.0)
    clip_values = np.full(raw.shape[1], np.inf, dtype=float)
    q = cfg.winsor_quantile
    for j, col in enumerate(feature_cols):
        if col != "Recency" and 0.90 <= q < 1.0:
            clip_values[j] = float(np.quantile(raw[:, j], q))

    X_fit = _feature_matrix(fit_rfm, feature_cols, clip_values)
    scaler = StandardScaler().fit(X_fit)
    X_scaled = scaler.transform(X_fit).astype(np.float32)

    model = MiniBatchKMeans(
        n_clusters=cfg.n_clusters,
        random_state=cfg.random_state,
        batch_size=cfg.batch_size,
        n_init=cfg.n_init,
        max_iter=cfg.max_iter,
        reassignment_ratio=0.01,
    ).fit(X_scaled)

    model.rfm_feature_cols_ = tuple(feature_cols)
    model.rfm_clip_values_ = clip_values
    model.rfm_active_only_ = bool(active_only)
    return model, scaler


def predict_with_reference(rfm, model, scaler):
    feature_cols = list(model.rfm_feature_cols_)
    X = _feature_matrix(rfm, feature_cols, model.rfm_clip_values_)
    X_scaled = scaler.transform(X).astype(np.float32)

    if model.rfm_active_only_:
        active = rfm["Frequency"].to_numpy() > 0
        labels = np.full(len(rfm), -1, dtype=np.int16)
        if active.any():
            labels[active] = model.predict(X_scaled[active]).astype(np.int16)
    else:
        labels = model.predict(X_scaled).astype(np.int16)

    out = rfm.copy()
    out["Cluster"] = labels
    return out, X_scaled, labels

print("Clustering functions defined.")

## Cell 8 — Evaluation metrics: population-aware + predictive proxy tasks

In [ ]:
PREDICTION_CACHE = {}


def internal_metrics(X, labels):
    X = np.asarray(X, dtype=np.float32)
    labels = np.asarray(labels)
    unique = np.unique(labels)
    if len(X) < 10 or len(unique) < 2 or len(X) <= len(unique):
        return {"silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "metric_n": int(len(X)),
                "metric_status": "insufficient_rows_or_clusters"}
    try:
        return {
            "silhouette": float(silhouette_score(X, labels)),
            "davies_bouldin": float(davies_bouldin_score(X, labels)),
            "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
            "metric_n": int(len(X)),
            "metric_status": "ok",
        }
    except Exception as exc:
        return {"silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "metric_n": int(len(X)),
                "metric_status": f"error:{type(exc).__name__}"}


def make_common_sample_ids(daily_agg, snapshot, max_n=3000, seed=SEED):
    ids = np.sort(daily_agg.loc[
        daily_agg["Date"] < pd.Timestamp(snapshot).normalize(), "CustomerID"
    ].unique())
    if len(ids) <= max_n:
        return ids
    day_seed = int(pd.Timestamp(snapshot).strftime("%Y%m%d")) % (2**32 - 1)
    rng = np.random.default_rng(seed + day_seed)
    return np.sort(rng.choice(ids, size=max_n, replace=False))


def metric_subset(rfm, X, labels, sample_ids, active_only=False):
    mask = rfm["CustomerID"].isin(sample_ids).to_numpy()
    if active_only:
        mask &= (rfm["Frequency"].to_numpy() > 0)
    return X[mask], labels[mask], int(mask.sum())


def _macro_direct_jaccard(a, b):
    labels = sorted(set(a).union(set(b)))
    vals = []
    for lab in labels:
        ma, mb = (a == lab), (b == lab)
        union = np.logical_or(ma, mb).sum()
        if union:
            vals.append(float(np.logical_and(ma, mb).sum() / union))
    return float(np.mean(vals)) if vals else np.nan


def stability_metrics(prev, curr):
    """Temporal stability on common customers; active-cluster stability is primary.

    prev/curr are DataFrames indexed by CustomerID with Cluster and Active columns.
    `ari_assignment_all` includes all-customer labels only when both models cluster all customers.
    `ari_active_both` compares cluster labels only among customers active in both snapshots.
    """
    common = prev.index.intersection(curr.index)
    base = {
        "n_common": int(len(common)), "n_active_both": 0,
        "ari_assignment_all": np.nan, "jaccard_assignment_all": np.nan,
        "ari_active_both": np.nan, "jaccard_active_both": np.nan,
        "active_to_inactive_rate": np.nan, "inactive_to_active_rate": np.nan,
    }
    if len(common) < 50:
        return base

    p = prev.loc[common]
    c = curr.loc[common]
    pa = p["Active"].to_numpy(dtype=bool)
    ca = c["Active"].to_numpy(dtype=bool)
    base["active_to_inactive_rate"] = float(np.mean(pa & ~ca))
    base["inactive_to_active_rate"] = float(np.mean(~pa & ca))

    # All-assignment stability is only meaningful when both snapshots use regular all-customer labels.
    if not (p["Cluster"].eq(-1).any() or c["Cluster"].eq(-1).any()):
        a_all = p["Cluster"].to_numpy()
        b_all = c["Cluster"].to_numpy()
        base["ari_assignment_all"] = float(adjusted_rand_score(a_all, b_all))
        base["jaccard_assignment_all"] = _macro_direct_jaccard(a_all, b_all)

    both_active = pa & ca
    base["n_active_both"] = int(both_active.sum())
    if both_active.sum() >= 50:
        a = p.loc[both_active, "Cluster"].to_numpy()
        b = c.loc[both_active, "Cluster"].to_numpy()
        # -1 is not allowed in this subset; guard against inconsistent active flags.
        valid = (a >= 0) & (b >= 0)
        a, b = a[valid], b[valid]
        if len(a) >= 50 and len(np.unique(a)) >= 2 and len(np.unique(b)) >= 2:
            base["ari_active_both"] = float(adjusted_rand_score(a, b))
            base["jaccard_active_both"] = _macro_direct_jaccard(a, b)
    return base


CLUSTER_VOCAB = [-1, 0, 1, 2, 3, 4]


def _prediction_features(rfm, feature_cols, *, use_cluster_feature=False):
    X = rfm[list(feature_cols)].to_numpy(dtype=float)
    X = np.maximum(np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0), 0.0)
    X = np.log1p(X)
    if use_cluster_feature and "Cluster" in rfm.columns:
        labels = rfm["Cluster"].astype(int).to_numpy()
        one_hot = np.column_stack([(labels == value).astype(float) for value in CLUSTER_VOCAB])
        X = np.hstack([X, one_hot])
    return X


def _binary_prediction_metrics(rfm_at_snapshot, full_data, snapshot,
                               horizon, feature_cols, label_name,
                               seed=SEED, use_cluster_feature=False):
    """Cross-sectional future-label discrimination at one snapshot (not temporal deployment)."""
    snapshot = pd.Timestamp(snapshot).normalize()
    horizon_end = snapshot + pd.Timedelta(days=int(horizon))
    data_end_excl = pd.Timestamp(full_data["InvoiceDate"].max()).normalize() + pd.Timedelta(days=1)
    result = {f"{label_name}_auc": np.nan, f"{label_name}_ap": np.nan,
              f"{label_name}_rate": np.nan, f"{label_name}_n": 0,
              f"{label_name}_status": "not_evaluated"}
    if horizon_end > data_end_excl:
        result[f"{label_name}_status"] = "insufficient_followup"
        return result

    future_ids = set(full_data.loc[
        (full_data["InvoiceDate"] >= snapshot) &
        (full_data["InvoiceDate"] < horizon_end), "CustomerID"
    ].unique())
    eval_df = rfm_at_snapshot.sort_values("CustomerID").dropna(subset=feature_cols).copy()
    y = eval_df["CustomerID"].isin(future_ids).astype("int8").to_numpy()
    result[f"{label_name}_n"] = int(len(y))
    result[f"{label_name}_rate"] = float(y.mean()) if len(y) else np.nan
    if len(y) < 100 or len(np.unique(y)) < 2:
        result[f"{label_name}_status"] = "insufficient_sample_or_class"
        return result
    if np.bincount(y, minlength=2).min() < 10:
        result[f"{label_name}_status"] = "too_few_minority_examples"
        return result

    X = _prediction_features(eval_df, feature_cols, use_cluster_feature=use_cluster_feature)
    idx = np.arange(len(y))
    tr, te = train_test_split(idx, test_size=0.30, random_state=seed, stratify=y)
    clf = GradientBoostingClassifier(
        n_estimators=60, learning_rate=0.05, max_depth=2, random_state=seed,
    )
    clf.fit(X[tr], y[tr])
    prob = clf.predict_proba(X[te])[:, 1]
    result[f"{label_name}_auc"] = float(roc_auc_score(y[te], prob))
    result[f"{label_name}_ap"] = float(average_precision_score(y[te], prob))
    result[f"{label_name}_status"] = "ok"
    return result


def future_purchase_metrics(rfm_at_snapshot, full_data, snapshot,
                            horizon=28, seed=SEED, feature_cols=RFM_RATE_COLS,
                            cache_key=None, use_cluster_feature=False):
    key = ("future_purchase", str(pd.Timestamp(snapshot).date()), int(horizon),
           tuple(feature_cols), str(cache_key), bool(use_cluster_feature))
    if key not in PREDICTION_CACHE:
        PREDICTION_CACHE[key] = _binary_prediction_metrics(
            rfm_at_snapshot, full_data, snapshot, horizon,
            feature_cols, "future_purchase", seed, use_cluster_feature,
        )
    return dict(PREDICTION_CACHE[key])


def churn_ground_truth(full_data, snapshot, churn_window=90, min_history=90):
    """Operational 90-day inactivity among customers active in prior min_history days."""
    snapshot = pd.Timestamp(snapshot).normalize()
    churn_end = snapshot + pd.Timedelta(days=int(churn_window))
    data_end = pd.Timestamp(full_data["InvoiceDate"].max()).normalize() + pd.Timedelta(days=1)
    if churn_end > data_end:
        return None
    cutoff = snapshot - pd.Timedelta(days=int(min_history))
    eligible = full_data.loc[
        (full_data["InvoiceDate"] >= cutoff) &
        (full_data["InvoiceDate"] < snapshot), "CustomerID"
    ].unique()
    bought = set(full_data.loc[
        (full_data["InvoiceDate"] >= snapshot) &
        (full_data["InvoiceDate"] < churn_end), "CustomerID"
    ].unique())
    labels = pd.Series(1, index=eligible, name="churn", dtype="int8")
    labels.loc[labels.index.isin(bought)] = 0
    return labels


def churn_auc_per_snapshot(rfm_at_snapshot, full_data, snapshot,
                           churn_window=90, seed=SEED, feature_cols=RFM_RATE_COLS,
                           cache_key=None, use_cluster_feature=False):
    key = ("churn", str(pd.Timestamp(snapshot).date()), int(churn_window),
           tuple(feature_cols), str(cache_key), bool(use_cluster_feature))
    if key not in PREDICTION_CACHE:
        labels = churn_ground_truth(full_data, snapshot, churn_window, min_history=90)
        if labels is None:
            result = {"churn_auc": np.nan, "churn_ap": np.nan, "churn_rate": np.nan,
                      "churn_n": 0, "churn_status": "insufficient_followup"}
        else:
            eval_df = rfm_at_snapshot.sort_values("CustomerID")
            eval_df = eval_df[eval_df["CustomerID"].isin(labels.index)].dropna(subset=feature_cols)
            y = labels.loc[eval_df["CustomerID"]].to_numpy()
            result = {"churn_auc": np.nan, "churn_ap": np.nan,
                      "churn_rate": float(y.mean()) if len(y) else np.nan,
                      "churn_n": int(len(y)), "churn_status": "ok"}
            if len(y) < 100 or len(np.unique(y)) < 2:
                result["churn_status"] = "insufficient_sample_or_class"
            elif np.bincount(y, minlength=2).min() < 10:
                result["churn_status"] = "too_few_minority_examples"
            else:
                X = _prediction_features(eval_df, feature_cols, use_cluster_feature=use_cluster_feature)
                idx = np.arange(len(y))
                tr, te = train_test_split(idx, test_size=0.30, random_state=seed, stratify=y)
                clf = GradientBoostingClassifier(
                    n_estimators=60, learning_rate=0.05, max_depth=2, random_state=seed,
                ).fit(X[tr], y[tr])
                prob = clf.predict_proba(X[te])[:, 1]
                result["churn_auc"] = float(roc_auc_score(y[te], prob))
                result["churn_ap"] = float(average_precision_score(y[te], prob))
        PREDICTION_CACHE[key] = result
    return dict(PREDICTION_CACHE[key])

print("Metrics functions defined. Prediction AUC/AP are cross-sectional proxy tasks.")

## Cell 9 — Temporal split, matched fixed baselines, experiment matrix

In [ ]:
start = pd.Timestamp(df["InvoiceDate"].min()).normalize()
end = pd.Timestamp(df["InvoiceDate"].max()).normalize()
total_days = (end - start).days
train_cutoff = start + pd.Timedelta(days=int(total_days * 0.70))
data_end_exclusive = end + pd.Timedelta(days=1)

print(f"Train cutoff (exclusive): {train_cutoff.date()}")
print(f"Train: {start.date()} → {(train_cutoff - pd.Timedelta(days=1)).date()}")
print(f"Holdout: {train_cutoff.date()} → {end.date()}")

# Primary comparisons: normalized RFM, all-customer reference clustering.
# Matched fixed windows bracket the adaptive distribution (median ~98d, mean ~110d, max 180d).
configs = {
    "fixed_4w": RFMParams(window_days=28, feature_mode="rate"),
    "fixed_8w": RFMParams(window_days=56, feature_mode="rate"),
    "fixed_13w": RFMParams(window_days=91, feature_mode="rate"),
    "fixed_14w": RFMParams(window_days=98, feature_mode="rate"),
    "fixed_16w": RFMParams(window_days=112, feature_mode="rate"),
    "fixed_26w": RFMParams(window_days=182, feature_mode="rate"),
    "adaptive_legacy": RFMParams(
        window_days=0, adaptive_method="individual", adaptive_multiplier=2.0,
        adaptive_min=28, adaptive_max=91, adaptive_default=56,
        adaptive_lookback=365, feature_mode="rate",
    ),
    "adaptive": RFMParams(
        window_days=0, adaptive_method="hierarchical", adaptive_multiplier=2.0,
        adaptive_min=28, adaptive_max=180, adaptive_default=56,
        adaptive_lookback=365, adaptive_shrinkage_k=3.0,
        adaptive_min_cohort_n=20, feature_mode="rate",
    ),
    "adaptive_wide": RFMParams(
        window_days=0, adaptive_method="hierarchical", adaptive_multiplier=2.0,
        adaptive_min=14, adaptive_max=180, adaptive_default=56,
        adaptive_lookback=365, adaptive_shrinkage_k=3.0,
        adaptive_min_cohort_n=20, feature_mode="rate",
    ),
    # Ablations: do not mix with the primary normalized/all-population leaderboard.
    "adaptive_raw": RFMParams(
        window_days=0, adaptive_method="hierarchical", adaptive_multiplier=2.0,
        adaptive_min=28, adaptive_max=180, adaptive_default=56,
        adaptive_lookback=365, adaptive_shrinkage_k=3.0,
        adaptive_min_cohort_n=20, feature_mode="raw",
    ),
    "fixed_13w_raw": RFMParams(window_days=91, feature_mode="raw"),
    "adaptive_two_stage": RFMParams(
        window_days=0, adaptive_method="hierarchical", adaptive_multiplier=2.0,
        adaptive_min=28, adaptive_max=180, adaptive_default=56,
        adaptive_lookback=365, adaptive_shrinkage_k=3.0,
        adaptive_min_cohort_n=20, feature_mode="rate", active_only=True,
    ),
}

PRIMARY_CONFIGS = [
    "fixed_4w", "fixed_8w", "fixed_13w", "fixed_14w", "fixed_16w", "fixed_26w",
    "adaptive_legacy", "adaptive", "adaptive_wide",
]
ACTIVE_ONLY_CONFIGS = ["adaptive_two_stage"]
ABLATION_CONFIGS = ["adaptive_raw", "fixed_13w_raw", "adaptive_two_stage"]
MATCHED_FIXED_CONFIGS = ["fixed_14w", "fixed_16w", "fixed_26w"]


def snapshot_dates(s, e, warmup=0, step=14):
    first = pd.Timestamp(s).normalize() + pd.Timedelta(days=int(warmup))
    last = pd.Timestamp(e).normalize()
    if first > last:
        return []
    dates = list(pd.date_range(first, last, freq=f"{step}D"))
    if not dates or dates[-1] != last:
        dates.append(last)
    return dates


df_train = df.loc[df["InvoiceDate"] < train_cutoff].reset_index(drop=True)
df_hold = df.loc[df["InvoiceDate"] >= train_cutoff].reset_index(drop=True)
if df_train.empty or df_hold.empty:
    raise ValueError("Temporal split created an empty train or holdout partition.")

train_dates = snapshot_dates(
    df_train["InvoiceDate"].min(), df_train["InvoiceDate"].max(), warmup=180, step=28
)
# Start holdout evaluation at the cutoff. Snapshot features only use dates < snapshot.
hold_dates = snapshot_dates(train_cutoff, df_hold["InvoiceDate"].max(), warmup=0, step=14)

daily_agg_full = precompute_daily(df)
daily_agg_train = precompute_daily(df_train)

SNAPSHOT_STEP = 14
MAX_ADAPTIVE_WINDOW = max(p.adaptive_max for p in configs.values() if p.window_days == 0)
MAX_COMPARISON_WINDOW = max(MAX_ADAPTIVE_WINDOW, max(
    p.window_days for p in configs.values() if p.window_days > 0
))
BLOCK_LENGTH = int(np.ceil(MAX_COMPARISON_WINDOW / SNAPSHOT_STEP))

REPRO["configs"] = {k: asdict(v) for k, v in configs.items()}
REPRO["comparison_groups"] = {
    "primary_normalized_all_customer": PRIMARY_CONFIGS,
    "active_only_ablation": ACTIVE_ONLY_CONFIGS,
    "raw_feature_ablation": ["adaptive_raw", "fixed_13w_raw"],
    "matched_fixed_windows_days": [98, 112, 182],
}
REPRO["hyperparams_derived"] = {
    "train_cutoff": str(train_cutoff.date()),
    "n_train_snapshots": len(train_dates),
    "n_holdout_snapshots": len(hold_dates),
    "max_adaptive_window": MAX_ADAPTIVE_WINDOW,
    "max_comparison_window": MAX_COMPARISON_WINDOW,
    "snapshot_step": SNAPSHOT_STEP,
    "block_length_primary": BLOCK_LENGTH,
}

print(f"Train snapshots: {len(train_dates)} | Holdout snapshots: {len(hold_dates)}")
print(f"Daily aggregates: full={len(daily_agg_full):,}, train={len(daily_agg_train):,}")
print("Primary comparable configs:", PRIMARY_CONFIGS)
print(f"Primary bootstrap block length: {BLOCK_LENGTH} snapshots")
print("Fixed-window matches: 98d, 112d, 182d")

## Cell 10 — Main experiments: active/all metrics, primary + cluster-feature ablation

In [ ]:
output_dir = Path("/content/rfm_outputs_v61")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

cluster_cfg = ClusteringConfig(n_clusters=5, metric_sample=2000, winsor_quantile=0.995)
PREDICTION_CACHE.clear()
FUTURE_HORIZON = 28
CHURN_HORIZON = 90
ref_snapshot = pd.Timestamp(df_train["InvoiceDate"].max()).normalize() + pd.Timedelta(days=1)
all_results = {}
REFERENCE_MODELS = {}
HOLDOUT_CACHE = {}


def _safe_mean(rows, key):
    vals = [float(r[key]) for r in rows
            if key in r and r[key] is not None and pd.notna(r[key])]
    return float(np.mean(vals)) if vals else np.nan


def _metric_for_snapshot(rfm, X, labels, sample_ids, model):
    sample_mask = rfm["CustomerID"].isin(sample_ids).to_numpy()
    active_mask = sample_mask & (rfm["Frequency"].to_numpy() > 0)

    active_metrics = internal_metrics(X[active_mask], labels[active_mask])
    active_result = {
        "silhouette_active": active_metrics["silhouette"],
        "dbi_active": active_metrics["davies_bouldin"],
        "ch_active": active_metrics["calinski_harabasz"],
        "n_active_metric": active_metrics["metric_n"],
        "active_metric_status": active_metrics["metric_status"],
    }

    if model.rfm_active_only_:
        all_result = {
            "silhouette_all": np.nan, "dbi_all": np.nan, "ch_all": np.nan,
            "n_all_metric": int(sample_mask.sum()),
            "all_metric_status": "not_comparable_active_only_model",
        }
    else:
        all_metrics = internal_metrics(X[sample_mask], labels[sample_mask])
        all_result = {
            "silhouette_all": all_metrics["silhouette"],
            "dbi_all": all_metrics["davies_bouldin"],
            "ch_all": all_metrics["calinski_harabasz"],
            "n_all_metric": all_metrics["metric_n"],
            "all_metric_status": all_metrics["metric_status"],
        }

    return {
        **active_result, **all_result,
        "metric_population_primary": "active_in_window",
        "active_rate_full": float((rfm["Frequency"] > 0).mean()),
        "inactive_rate_full": float((rfm["Frequency"] == 0).mean()),
    }


def _label_state(labeled):
    state = labeled.set_index("CustomerID")[["Cluster", "Frequency"]].copy()
    state["Active"] = state["Frequency"] > 0
    return state[["Cluster", "Active"]]


for name, params in configs.items():
    config_start_time = time.perf_counter()
    LOG.info("Experiment: %s | features=%s | active_only=%s",
             name, params.feature_mode, params.active_only)
    cfg_dir = output_dir / name
    cfg_dir.mkdir(parents=True, exist_ok=True)
    feature_cols = RFM_COLS.copy() if params.feature_mode == "raw" else RFM_RATE_COLS.copy()

    # Reference model is fit using train data only; its scaler/centroids remain fixed in holdout.
    ref_rfm = compute_windowed_rfm(df_train, ref_snapshot, params, daily_agg=daily_agg_train)
    assert_rfm_invariants(ref_rfm, params)
    model, scaler = fit_reference(
        ref_rfm, cluster_cfg, feature_cols=feature_cols, active_only=params.active_only
    )
    ref_labeled, ref_X, ref_labels = predict_with_reference(ref_rfm, model, scaler)
    REFERENCE_MODELS[name] = (model, scaler)
    ref_sample = make_common_sample_ids(
        daily_agg_train, ref_snapshot, cluster_cfg.metric_sample, seed=SEED
    )
    ref_metrics = _metric_for_snapshot(ref_rfm, ref_X, ref_labels, ref_sample, model)
    ref_labeled.to_csv(cfg_dir / "reference_segments.csv", index=False)

    # Train diagnostics are retrospective because the reference model is fit at end of train.
    train_rows = []
    prev_train_state = None
    for snap in train_dates:
        rfm = compute_windowed_rfm(df_train, snap, params, daily_agg=daily_agg_train)
        if len(rfm) < 50:
            continue
        labeled, X, labels = predict_with_reference(rfm, model, scaler)
        sample_ids = make_common_sample_ids(
            daily_agg_train, snap, cluster_cfg.metric_sample, seed=SEED
        )
        metrics = _metric_for_snapshot(rfm, X, labels, sample_ids, model)
        curr_state = _label_state(labeled)
        stability = stability_metrics(prev_train_state, curr_state) if prev_train_state is not None else {}
        prev_train_state = curr_state
        train_rows.append({"snapshot": snap.date(), "n_customers": len(rfm), **metrics, **stability})

    pd.DataFrame(train_rows).to_csv(cfg_dir / "train_metrics.csv", index=False)

    # Holdout metrics: use the same snapshot sample IDs across every configuration.
    hold_rows_primary, hold_rows_ablation = [], []
    prev_hold_state = None
    for snap in hold_dates:
        rfm = compute_windowed_rfm(df, snap, params, daily_agg=daily_agg_full)
        if len(rfm) < 50:
            continue
        labeled, X, labels = predict_with_reference(rfm, model, scaler)
        HOLDOUT_CACHE[(name, snap.date())] = (labeled, X, labels)
        sample_ids = make_common_sample_ids(
            daily_agg_full, snap, cluster_cfg.metric_sample, seed=SEED
        )
        metrics = _metric_for_snapshot(rfm, X, labels, sample_ids, model)
        curr_state = _label_state(labeled)
        stability = stability_metrics(prev_hold_state, curr_state) if prev_hold_state is not None else {}
        prev_hold_state = curr_state
        common_row = {
            "snapshot": snap.date(), "n_customers": len(rfm),
            "feature_mode": params.feature_mode,
            "active_only_model": params.active_only,
            "fit_population": "active_only" if params.active_only else "all_customers",
            **metrics, **stability,
        }

        # Prediction proxy is RFM-only. For Two-stage, clustering method does not enter X;
        # mark it as not applicable rather than presenting a duplicate score as a method gain.
        if params.active_only:
            fut_p = {"future_purchase_auc": np.nan, "future_purchase_ap": np.nan,
                     "future_purchase_rate": np.nan, "future_purchase_n": 0,
                     "future_purchase_status": "not_applicable_segmentation_only"}
            chu_p = {"churn_auc": np.nan, "churn_ap": np.nan, "churn_rate": np.nan,
                     "churn_n": 0, "churn_status": "not_applicable_segmentation_only"}
        else:
            fut_p = future_purchase_metrics(
                labeled, df, snap, horizon=FUTURE_HORIZON, seed=SEED,
                feature_cols=feature_cols, cache_key=name, use_cluster_feature=False,
            )
            chu_p = churn_auc_per_snapshot(
                labeled, df, snap, churn_window=CHURN_HORIZON, seed=SEED,
                feature_cols=feature_cols, cache_key=name, use_cluster_feature=False,
            )
        hold_rows_primary.append({**common_row, **fut_p, **chu_p})

        # Secondary representation ablation; never used in primary model ranking.
        fut_a = future_purchase_metrics(
            labeled, df, snap, horizon=FUTURE_HORIZON, seed=SEED,
            feature_cols=feature_cols, cache_key=f"{name}_withcluster", use_cluster_feature=True,
        )
        chu_a = churn_auc_per_snapshot(
            labeled, df, snap, churn_window=CHURN_HORIZON, seed=SEED,
            feature_cols=feature_cols, cache_key=f"{name}_withcluster", use_cluster_feature=True,
        )
        hold_rows_ablation.append({**common_row, **fut_a, **chu_a})

    pd.DataFrame(hold_rows_primary).to_csv(cfg_dir / "holdout_metrics.csv", index=False)
    pd.DataFrame(hold_rows_ablation).to_csv(cfg_dir / "holdout_metrics_withcluster.csv", index=False)

    all_results[name] = {
        "params": asdict(params),
        "feature_cols": feature_cols,
        "reference_metrics": ref_metrics,
        "n_train_snapshots": len(train_rows),
        "n_holdout_snapshots": len(hold_rows_primary),
        "n_valid_future_auc": sum(r.get("future_purchase_status") == "ok" for r in hold_rows_primary),
        "n_valid_churn_auc": sum(r.get("churn_status") == "ok" for r in hold_rows_primary),
        "runtime_seconds": float(time.perf_counter() - config_start_time),
    }
    LOG.info("Runtime %s: %.1f sec", name, all_results[name]["runtime_seconds"])

REPRO["runtime_seconds_by_config"] = {
    k: v["runtime_seconds"] for k, v in all_results.items()
}
(output_dir / "metadata.json").write_text(
    json.dumps(all_results, indent=2, default=str), encoding="utf-8"
)
(output_dir / "reproducibility.json").write_text(
    json.dumps(REPRO, indent=2, default=str), encoding="utf-8"
)
print(f"All configured experiments completed: {len(all_results)}/{len(configs)}")

## Cell 11 — Summary, descriptive uncertainty, and coverage

In [ ]:
def noncircular_moving_block_ci(values, block_length, n_boot=2000, seed=SEED, alpha=0.05):
    """Non-circular moving-block bootstrap on the original time index.

    NaNs remain at their original positions. A block is eligible only if every
    time position in it is finite. CI is withheld when fewer than 3 valid start
    positions exist, because the effective temporal support is too small.
    """
    x = np.asarray(values, dtype=float)
    n_total = len(x)
    finite = np.isfinite(x)
    mean_value = float(np.nanmean(x)) if finite.any() else np.nan
    L = int(max(1, block_length))
    starts = [s for s in range(0, n_total - L + 1) if np.isfinite(x[s:s+L]).all()]
    if finite.sum() < 5 or len(starts) < 3:
        return {"mean": mean_value, "ci_low": np.nan, "ci_high": np.nan,
                "n_valid": int(finite.sum()), "n_timeline": n_total,
                "block_length": L, "n_valid_block_starts": len(starts),
                "ci_status": "insufficient_contiguous_block_support"}
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n_total / L))
    boots = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        sample = np.concatenate([x[s:s+L] for s in chosen])[:n_total]
        boots[b] = float(np.mean(sample))
    lo, hi = np.quantile(boots, [alpha/2, 1-alpha/2])
    return {"mean": mean_value, "ci_low": float(lo), "ci_high": float(hi),
            "n_valid": int(finite.sum()), "n_timeline": n_total,
            "block_length": L, "n_valid_block_starts": len(starts),
            "ci_status": "exploratory_block_bootstrap"}


metric_cols = [
    "silhouette_active", "dbi_active", "ch_active",
    "silhouette_all", "dbi_all", "ch_all",
    "ari_active_both", "jaccard_active_both",
    "ari_assignment_all", "jaccard_assignment_all",
    "future_purchase_auc", "future_purchase_ap",
    "churn_auc", "churn_ap",
]
summary_rows, ci_rows = [], []
for cfg in configs:
    h_path = output_dir / cfg / "holdout_metrics.csv"
    if not h_path.exists():
        continue
    h = pd.read_csv(h_path).set_index("snapshot")
    params = configs[cfg]
    row = {
        "config": cfg,
        "feature_mode": params.feature_mode,
        "fit_population": "active_only" if params.active_only else "all_customers",
        "metric_population_primary": "active_in_window",
        "window_spec": "adaptive" if params.window_days == 0 else f"fixed_{params.window_days}d",
        "n_holdout_snapshots": int(len(h)),
    }
    for metric in metric_cols:
        if metric not in h.columns:
            continue
        vals = pd.to_numeric(h[metric], errors="coerce")
        valid = vals.dropna()
        row[f"{metric}_mean"] = float(valid.mean()) if len(valid) else np.nan
        row[f"{metric}_std"] = float(valid.std(ddof=1)) if len(valid) > 1 else np.nan
        row[f"{metric}_valid_n"] = int(len(valid))
        timeline = np.array([
            float(h.loc[str(snap.date()), metric])
            if str(snap.date()) in h.index and pd.notna(h.loc[str(snap.date()), metric])
            else np.nan
            for snap in hold_dates
        ], dtype=float)
        ci = noncircular_moving_block_ci(
            timeline, BLOCK_LENGTH, n_boot=2000, seed=SEED
        )
        ci_rows.append({"config": cfg, "metric": metric, **ci})
    row["future_purchase_valid_n"] = int((h.get("future_purchase_status", pd.Series(index=h.index)) == "ok").sum())
    row["churn_valid_n"] = int((h.get("churn_status", pd.Series(index=h.index)) == "ok").sum())
    row["runtime_seconds"] = all_results.get(cfg, {}).get("runtime_seconds", np.nan)
    summary_rows.append(row)

summary_full = pd.DataFrame(summary_rows)
summary_ci = pd.DataFrame(ci_rows)
summary_full.to_csv(output_dir / "summary.csv", index=False)
summary_ci.to_csv(output_dir / "summary_block_ci.csv", index=False)

print("PRIMARY SUMMARY — active-in-window clustering metrics; no composite score")
show_cols = [
    "config", "feature_mode", "fit_population", "n_holdout_snapshots",
    "silhouette_active_mean", "dbi_active_mean", "ch_active_mean",
    "ari_active_both_mean", "jaccard_active_both_mean",
    "future_purchase_auc_mean", "future_purchase_ap_mean",
    "future_purchase_valid_n", "churn_auc_mean", "churn_valid_n",
]
display(summary_full.reindex(columns=show_cols).round(4))
print("\nBlock CI status (CI is withheld when too few contiguous blocks are available):")
display(summary_ci.groupby(["metric", "ci_status"], dropna=False)
        .size().rename("config_count").reset_index())
print("\nChurn is exploratory only; it is not used for primary ranking.")

Cell 12 — Adaptive window diagnostics (giữ nguyên v5)


In [ ]:
print("=" * 70)
print("HIERARCHICAL ADAPTIVE WINDOW DIAGNOSTICS")
print("=" * 70)

for cfg_name in ["adaptive_legacy", "adaptive", "adaptive_wide"]:
    if cfg_name not in configs:
        continue
    params = configs[cfg_name]
    rfm_diag = compute_windowed_rfm(df_train, ref_snapshot, params, daily_agg=daily_agg_train)
    n = len(rfm_diag)
    fallback_rate = rfm_diag["AdaptiveFallback"].mean() if n else np.nan
    print(f"\n{cfg_name}: customers={n:,}, no-personal-gap={fallback_rate:.1%}")
    print("Window distribution:")
    display(rfm_diag["Window"].describe(percentiles=[.1, .25, .5, .75, .9]).to_frame())
    print("Source distribution:")
    display(rfm_diag["AdaptiveSource"].value_counts(dropna=False).to_frame("customers"))
    print("Boundary rates:")
    print(f"  at min={np.mean(rfm_diag['Window'] == params.adaptive_min):.1%}; "
          f"at max={np.mean(rfm_diag['Window'] == params.adaptive_max):.1%}")

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(rfm_diag["Window"], bins=25, edgecolor="black", alpha=0.75)
    ax.set_title(f"{cfg_name}: adaptive window distribution")
    ax.set_xlabel("Window days")
    ax.set_ylabel("Customers")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

Cell 13 — Holdout evolution plots (giữ nguyên v5)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for name in configs:
    hpath = output_dir / name / "holdout_metrics.csv"
    if hpath.exists():
        h = pd.read_csv(hpath)
        if not h.empty:
            h["snapshot"] = pd.to_datetime(h["snapshot"])
            axes[0, 0].plot(h["snapshot"], h["silhouette"], marker=".", label=name)
            axes[0, 1].plot(h["snapshot"], h["ari"], marker=".", label=name)
            axes[1, 0].plot(h["snapshot"], h["future_purchase_auc"], marker=".", label=name)
            axes[1, 1].plot(h["snapshot"], h["inactive_in_window_rate"], marker=".", label=name)

axes[0, 0].set_title("Holdout Silhouette")
axes[0, 1].set_title("Holdout ARI")
axes[1, 0].set_title("Future-purchase AUC (no cluster feature)")
axes[1, 1].set_title("Inactive-in-window share")
for ax in axes.flat:
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

Cell 14 — Literature comparison template


In [ ]:
literature_schema = pd.DataFrame(columns=[
    "paper", "dataset", "population", "features", "preprocessing",
    "algorithm", "k", "metric", "value", "source_section",
    "comparable", "notes",
])
print("Literature comparison template (empty until verified):")
display(literature_schema)

Cell 15 — Cluster profiles (giữ nguyên v5)


In [ ]:
print("=" * 70)
print("CLUSTER PROFILES")
print("=" * 70)

for name in configs:
    ref_path = output_dir / name / "reference_segments.csv"
    if not ref_path.exists():
        continue
    ref = pd.read_csv(ref_path)
    prof = (
        ref.groupby("Cluster")
        .agg(
            n=("CustomerID", "count"),
            inactive_share=("Frequency", lambda x: float((x == 0).mean())),
            R_mean=("Recency", "mean"), R_median=("Recency", "median"),
            F_mean=("Frequency", "mean"), F_median=("Frequency", "median"),
            M_mean=("Monetary", "mean"), M_median=("Monetary", "median"),
            F30_mean=("FrequencyRate30", "mean"),
            M30_mean=("MonetaryRate30", "mean"),
        )
    )
    prof["share"] = prof["n"] / prof["n"].sum()

    def profile_hint(row):
        if int(row.name) == -1:
            return "Inactive-in-window (two-stage)"
        if row["inactive_share"] >= 0.80:
            return "Inactive-heavy cluster"
        if row["R_mean"] <= prof["R_mean"].quantile(0.25) and row["F30_mean"] >= prof["F30_mean"].quantile(0.75):
            return "Recent high-frequency"
        if row["M30_mean"] >= prof["M30_mean"].quantile(0.75):
            return "High monetary rate"
        if row["R_mean"] >= prof["R_mean"].quantile(0.75):
            return "High recency / at-risk candidate"
        if row["F30_mean"] <= prof["F30_mean"].quantile(0.25):
            return "Low purchase rate"
        return "Middle activity"

    prof["descriptive_hint"] = prof.apply(profile_hint, axis=1)
    print(f"\n=== {name} ===")
    display(prof.round(3))

## Cell 16 — Separate leaderboards and exploratory paired tests (no overall rank-sum)

In [ ]:
def holm_bonferroni(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


def paired_wilcoxon_table(holdout_dir, config_pairs, metrics):
    """Exploratory snapshot-paired Wilcoxon; temporal dependence caveat is explicit."""
    rows = []
    for metric in metrics:
        metric_rows = []
        for a, b in config_pairs:
            pa = holdout_dir / a / "holdout_metrics.csv"
            pb = holdout_dir / b / "holdout_metrics.csv"
            if not pa.exists() or not pb.exists():
                continue
            da = pd.read_csv(pa).set_index("snapshot")
            db = pd.read_csv(pb).set_index("snapshot")
            common = da.index.intersection(db.index)
            if metric not in da.columns or metric not in db.columns:
                continue
            pair = da.loc[common, metric].to_frame("a").join(
                db.loc[common, metric].to_frame("b")).dropna()
            if len(pair) < 10:
                continue
            diff = pair["a"] - pair["b"]
            try:
                stat, p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
            except ValueError:
                continue
            metric_rows.append({
                "metric": metric, "comparison": f"{a} - {b}",
                "n_pairs": int(len(pair)), "median_difference": float(diff.median()),
                "mean_difference": float(diff.mean()), "statistic": float(stat),
                "p_raw": float(p),
                "interpretation_status": "exploratory_temporally_dependent_snapshots",
            })
        if metric_rows:
            adj = holm_bonferroni([r["p_raw"] for r in metric_rows])
            for row, p_adj in zip(metric_rows, adj):
                row["p_holm"] = float(p_adj)
                row["significant_holm_0.05_exploratory"] = bool(p_adj < 0.05)
            rows.extend(metric_rows)
    return pd.DataFrame(rows)


print("SEPARATE LEADERBOARDS — no composite rank-sum")

# Primary normalized all-customer reference methods only.
primary = summary_full[
    summary_full["config"].isin(PRIMARY_CONFIGS)
    & summary_full["feature_mode"].eq("rate")
    & summary_full["fit_population"].eq("all_customers")
].copy()

quality_metrics = [
    ("silhouette_active_mean", "max"),
    ("dbi_active_mean", "min"),
    ("ch_active_mean", "max"),
]
stability_metric_group = [
    ("ari_active_both_mean", "max"),
    ("jaccard_active_both_mean", "max"),
]
prediction_metric_group = [
    ("future_purchase_auc_mean", "max"),
    ("future_purchase_ap_mean", "max"),
]

def print_metric_leaderboard(frame, specs, title):
    print("\n" + title)
    rows = []
    for metric, direction in specs:
        if metric not in frame.columns:
            continue
        valid = frame[["config", metric]].dropna()
        if valid.empty:
            continue
        order = valid.sort_values(metric, ascending=(direction == "min"))
        for rank, (_, r) in enumerate(order.iterrows(), start=1):
            rows.append({"metric": metric, "rank": rank, "config": r["config"], "value": r[metric]})
    display(pd.DataFrame(rows).round(4))

print_metric_leaderboard(primary, quality_metrics, "A) CLUSTERING QUALITY — evaluated on active-in-window customers")
print_metric_leaderboard(primary, stability_metric_group, "B) TEMPORAL STABILITY — active in both consecutive snapshots")
print_metric_leaderboard(primary, prediction_metric_group, "C) PREDICTIVE UTILITY — cross-sectional future-purchase proxy")

# Separate all-population table; it is descriptive and not the primary segmentation score.
all_population = summary_full[summary_full["config"].isin(PRIMARY_CONFIGS)].copy()
print("\nD) SECONDARY ALL-POPULATION CLUSTERING METRICS")
display(all_population[["config", "silhouette_all_mean", "dbi_all_mean", "ch_all_mean"]]
        .sort_values("silhouette_all_mean", ascending=False).round(4))

# Active-only clustering is displayed separately: same evaluation population (active),
# but different fit population; never mix it into the all-customer primary leaderboard.
active_only_summary = summary_full[summary_full["config"].isin(ACTIVE_ONLY_CONFIGS)]
print("\nE) ACTIVE-ONLY MODEL — separate fit-population analysis")
display(active_only_summary[["config", "fit_population", "silhouette_active_mean",
                              "dbi_active_mean", "ch_active_mean",
                              "ari_active_both_mean", "jaccard_active_both_mean"]].round(4))

# Churn is always a separate exploratory table, never ranked or used in overall conclusion.
print("\nF) CHURN — EXPLORATORY ONLY; report valid snapshot count")
display(summary_full[["config", "churn_auc_mean", "churn_ap_mean", "churn_valid_n"]].round(4))

# Paired tests compare adaptive methods only against normalized fixed references on common snapshots.
comparison_pairs = [("adaptive", x) for x in
                    ["fixed_13w", "fixed_14w", "fixed_16w", "fixed_26w"]]
comparison_pairs += [("adaptive", "adaptive_legacy"), ("adaptive_wide", "adaptive")]
wilcoxon_df = paired_wilcoxon_table(
    output_dir, comparison_pairs,
    ["silhouette_active", "ari_active_both", "jaccard_active_both",
     "future_purchase_auc", "future_purchase_ap"],
)
wilcoxon_path = output_dir / "paired_wilcoxon_holm_exploratory.csv"
wilcoxon_df.to_csv(wilcoxon_path, index=False)
print("\nG) PAIRED WILCOXON + HOLM — exploratory only due temporal dependence")
if not wilcoxon_df.empty:
    display(wilcoxon_df.round(4))
else:
    print("No metric has at least 10 valid paired snapshots for the selected comparisons.")

print("\nNo global rank-sum is computed. Metric groups answer different research questions.")

## Cell 17 — Population sensitivity: all vs active-in-window (descriptive)

In [ ]:
print("POPULATION SENSITIVITY — all vs active-in-window")
population_rows = []
population_configs = ["fixed_13w", "fixed_16w", "adaptive_legacy", "adaptive", "adaptive_wide", "adaptive_two_stage"]
for name in population_configs:
    params = configs[name]
    for snap in hold_dates:
        cached = HOLDOUT_CACHE.get((name, snap.date()))
        if cached is None:
            continue
        rfm_snap, X, labels = cached
        sample_ids = make_common_sample_ids(daily_agg_full, snap, cluster_cfg.metric_sample, seed=SEED)
        all_x, all_y, n_all = metric_subset(rfm_snap, X, labels, sample_ids, active_only=False)
        act_x, act_y, n_active = metric_subset(rfm_snap, X, labels, sample_ids, active_only=True)
        all_m = internal_metrics(all_x, all_y) if not params.active_only else {
            "silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan
        }
        active_m = internal_metrics(act_x, act_y)
        population_rows.append({
            "config": name, "snapshot": snap.date(),
            "fit_population": "active_only" if params.active_only else "all_customers",
            "n_all_sample": n_all, "n_active_sample": n_active,
            "inactive_rate_full": float((rfm_snap["Frequency"] == 0).mean()),
            "silhouette_all": all_m.get("silhouette", np.nan),
            "silhouette_active": active_m["silhouette"],
            "dbi_all": all_m.get("davies_bouldin", np.nan),
            "dbi_active": active_m["davies_bouldin"],
        })

population_df = pd.DataFrame(population_rows)
population_df.to_csv(output_dir / "population_sensitivity.csv", index=False)
if not population_df.empty:
    population_summary = (population_df.groupby(["config", "fit_population"])
        .agg(inactive_rate_mean=("inactive_rate_full", "mean"),
             silhouette_all_mean=("silhouette_all", "mean"),
             silhouette_active_mean=("silhouette_active", "mean"),
             dbi_active_mean=("dbi_active", "mean"),
             valid_snapshots=("snapshot", "count"))
        .reset_index())
    display(population_summary.round(4))
    print("Do not compare silhouette_all against silhouette_active: populations differ.")

## Cell 18 — Horizon sensitivity (predictive proxy, no cluster feature)

In [ ]:
print("FUTURE-PURCHASE HORIZON SENSITIVITY — cross-sectional proxy")
horizons = [28, 56, 91]
horizon_configs = ["fixed_13w", "fixed_14w", "fixed_16w", "fixed_26w", "adaptive_legacy", "adaptive", "adaptive_wide"]

def horizon_snapshot_is_valid(snapshot, horizon):
    snapshot = pd.Timestamp(snapshot).normalize()
    horizon_end = snapshot + pd.Timedelta(days=int(horizon))
    if horizon_end > data_end_exclusive:
        return False
    current_ids = daily_agg_full.loc[daily_agg_full["Date"] < snapshot, "CustomerID"].unique()
    future_ids = set(df.loc[
        (df["InvoiceDate"] >= snapshot) & (df["InvoiceDate"] < horizon_end), "CustomerID"
    ].unique())
    y = np.isin(current_ids, list(future_ids)).astype("int8")
    return len(y) >= 100 and np.bincount(y, minlength=2).min() >= 10

horizon_rows = []
for name in horizon_configs:
    params = configs[name]
    feature_cols = RFM_COLS.copy() if params.feature_mode == "raw" else RFM_RATE_COLS.copy()
    for horizon in horizons:
        for snap in hold_dates:
            if not horizon_snapshot_is_valid(snap, horizon):
                continue
            cached = HOLDOUT_CACHE.get((name, snap.date()))
            if cached is None:
                continue
            rfm_h, _, _ = cached
            result = future_purchase_metrics(
                rfm_h.drop(columns=["Cluster"], errors="ignore"), df, snap,
                horizon=horizon, seed=SEED, feature_cols=feature_cols,
                cache_key=f"{name}_horizon_{horizon}_no_cluster",
                use_cluster_feature=False,
            )
            horizon_rows.append({
                "config": name, "snapshot": snap.date(), "horizon_days": horizon,
                **result,
            })

horizon_df = pd.DataFrame(horizon_rows)
horizon_df.to_csv(output_dir / "horizon_sensitivity.csv", index=False)
if not horizon_df.empty:
    horizon_summary = (horizon_df.groupby(["config", "horizon_days"])
        .agg(valid_snapshots=("future_purchase_status", lambda s: int((s == "ok").sum())),
             auc_mean=("future_purchase_auc", "mean"),
             auc_std=("future_purchase_auc", "std"),
             ap_mean=("future_purchase_ap", "mean"),
             purchase_rate_mean=("future_purchase_rate", "mean"))
        .reset_index())
    display(horizon_summary.round(4))
print("AUC/AP are future-label proxy tasks, not event-detection accuracy or temporal deployment results.")

## Cell 19 — Paired non-circular moving-block bootstrap (time index preserved)

In [ ]:
print(f"PAIRED NON-CIRCULAR BLOCK BOOTSTRAP — primary block={BLOCK_LENGTH}")
N_BOOT = 2000
BLOCK_SENSITIVITY = sorted(set([max(3, BLOCK_LENGTH // 2), BLOCK_LENGTH]))


def paired_non_circular_block_bootstrap(diff_timeline, block_length, n_boot=N_BOOT, seed=SEED):
    x = np.asarray(diff_timeline, dtype=float)
    n = len(x)
    finite = np.isfinite(x)
    observed = float(np.nanmean(x)) if finite.any() else np.nan
    L = int(block_length)
    valid_starts = [s for s in range(0, n - L + 1) if np.isfinite(x[s:s+L]).all()]
    if finite.sum() < 5 or len(valid_starts) < 3:
        return {"mean_difference": observed, "ci_low": np.nan, "ci_high": np.nan,
                "n_valid_pairs": int(finite.sum()), "n_timeline": n,
                "block_length": L, "n_valid_block_starts": len(valid_starts),
                "status": "insufficient_contiguous_temporal_support"}
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / L))
    boot = np.empty(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(valid_starts, size=n_blocks, replace=True)
        sample = np.concatenate([x[s:s+L] for s in chosen])[:n]
        boot[b] = float(np.mean(sample))
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return {"mean_difference": observed, "ci_low": float(lo), "ci_high": float(hi),
            "n_valid_pairs": int(finite.sum()), "n_timeline": n,
            "block_length": L, "n_valid_block_starts": len(valid_starts),
            "status": "exploratory_non_circular_block_bootstrap"}


metrics_for_bootstrap = [
    "silhouette_active", "ari_active_both", "jaccard_active_both",
    "future_purchase_auc", "future_purchase_ap",
]
pairs = [("adaptive", x) for x in ["fixed_13w", "fixed_14w", "fixed_16w", "fixed_26w"]]
pairs += [("adaptive", "adaptive_legacy"), ("adaptive_wide", "adaptive")]
bootstrap_rows = []
all_snapshot_keys = [str(pd.Timestamp(s).date()) for s in hold_dates]

for method_a, method_b in pairs:
    pa, pb = output_dir / method_a / "holdout_metrics.csv", output_dir / method_b / "holdout_metrics.csv"
    if not pa.exists() or not pb.exists():
        continue
    da, db = pd.read_csv(pa).set_index("snapshot"), pd.read_csv(pb).set_index("snapshot")
    for metric in metrics_for_bootstrap:
        if metric not in da.columns or metric not in db.columns:
            continue
        diff = np.full(len(all_snapshot_keys), np.nan, dtype=float)
        for i, snap_key in enumerate(all_snapshot_keys):
            if snap_key in da.index and snap_key in db.index:
                va, vb = da.loc[snap_key, metric], db.loc[snap_key, metric]
                if pd.notna(va) and pd.notna(vb):
                    diff[i] = float(va) - float(vb)
        for block_len in BLOCK_SENSITIVITY:
            stat = paired_non_circular_block_bootstrap(diff, block_len, seed=SEED + block_len)
            bootstrap_rows.append({
                "comparison": f"{method_a} - {method_b}", "metric": metric,
                "block_role": "primary" if block_len == BLOCK_LENGTH else "sensitivity",
                **stat,
            })

block_boot_df = pd.DataFrame(bootstrap_rows)
block_boot_df.to_csv(output_dir / "paired_block_bootstrap.csv", index=False)
display(block_boot_df.round(4))
print("NaN positions remain on the original snapshot timeline; no circular wrap is used.")
print("If there are too few contiguous blocks, CI is withheld rather than fabricated.")

## Cell 20 — Holdout segment transitions and active/inactive transitions

In [ ]:
print("HOLDOUT SEGMENT TRANSITIONS — fixed reference centroids")
transition_rows = []
for name, params in configs.items():
    prev_state = None
    for snap in hold_dates:
        cached = HOLDOUT_CACHE.get((name, snap.date()))
        if cached is None:
            continue
        labeled, _, _ = cached
        cur_state = _label_state(labeled)
        row = {"config": name, "snapshot": snap.date(),
               "n_customers": len(cur_state), "fit_population": "active_only" if params.active_only else "all_customers"}
        if prev_state is not None:
            row.update(stability_metrics(prev_state, cur_state))
        prev_state = cur_state
        transition_rows.append(row)

transition_df = pd.DataFrame(transition_rows)
transition_df.to_csv(output_dir / "holdout_transitions.csv", index=False)
display(transition_df.round(4))

# First-to-last matrices are descriptive, not a drift ground truth.
for name in configs:
    if not hold_dates:
        continue
    first = HOLDOUT_CACHE.get((name, hold_dates[0].date()))
    last = HOLDOUT_CACHE.get((name, hold_dates[-1].date()))
    if first is None or last is None:
        continue
    first_rfm, _, _ = first
    last_rfm, _, _ = last
    first_state, last_state = _label_state(first_rfm), _label_state(last_rfm)
    joined = first_state.join(last_state, how="inner", lsuffix="_first", rsuffix="_last")
    ct = pd.crosstab(joined["Cluster_first"], joined["Cluster_last"])
    print(f"\n{name}: first → last | shared customers={len(joined):,}")
    display(ct)
    active_transition = pd.crosstab(joined["Active_first"], joined["Active_last"], normalize="index")
    print("Active/inactive transition rates:")
    display(active_transition.round(3))

## Cell 21 — Final results package, outputs audit, and Excel workbook

In [ ]:
print("=" * 75)
print("FINAL EXPERIMENT RESULTS — v6.1")
print("=" * 75)

metric_cols = [
    "silhouette_active", "dbi_active", "ch_active",
    "silhouette_all", "ari_active_both", "jaccard_active_both",
    "future_purchase_auc", "future_purchase_ap", "churn_auc", "churn_ap",
]
all_holdout_parts, all_train_parts, summary_rows = [], [], []
for config_name in configs:
    cfg_dir = output_dir / config_name
    hp, tp = cfg_dir / "holdout_metrics.csv", cfg_dir / "train_metrics.csv"
    if hp.exists():
        h = pd.read_csv(hp)
        h.insert(0, "config", config_name)
        all_holdout_parts.append(h)
        row = {
            "config": config_name,
            "feature_mode": configs[config_name].feature_mode,
            "fit_population": "active_only" if configs[config_name].active_only else "all_customers",
            "n_holdout_snapshots": int(len(h)),
            "runtime_seconds": all_results.get(config_name, {}).get("runtime_seconds", np.nan),
        }
        for metric in metric_cols:
            if metric in h.columns:
                vals = pd.to_numeric(h[metric], errors="coerce").dropna()
                row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
                row[f"{metric}_std"] = float(vals.std(ddof=1)) if len(vals) > 1 else np.nan
                row[f"{metric}_valid_n"] = int(len(vals))
        row["future_purchase_valid_n"] = int((h.get("future_purchase_status", pd.Series(dtype=str)) == "ok").sum())
        row["churn_valid_n"] = int((h.get("churn_status", pd.Series(dtype=str)) == "ok").sum())
        summary_rows.append(row)
    if tp.exists():
        t = pd.read_csv(tp)
        t.insert(0, "config", config_name)
        all_train_parts.append(t)

holdout_all = pd.concat(all_holdout_parts, ignore_index=True) if all_holdout_parts else pd.DataFrame()
train_all = pd.concat(all_train_parts, ignore_index=True) if all_train_parts else pd.DataFrame()
final_summary = pd.DataFrame(summary_rows)

# Cluster-feature ablation: report the delta separately; it does not enter primary ranking.
ablation_rows = []
for cfg, params in configs.items():
    with_path = output_dir / cfg / "holdout_metrics_withcluster.csv"
    without_path = output_dir / cfg / "holdout_metrics.csv"
    if not with_path.exists() or not without_path.exists():
        continue
    h_with, h_without = pd.read_csv(with_path), pd.read_csv(without_path)
    def mean_metric(frame, metric):
        if metric not in frame.columns:
            return np.nan
        vals = pd.to_numeric(frame[metric], errors="coerce").dropna()
        return float(vals.mean()) if len(vals) else np.nan
    auc_with = mean_metric(h_with, "future_purchase_auc")
    auc_without = mean_metric(h_without, "future_purchase_auc")
    ap_with = mean_metric(h_with, "future_purchase_ap")
    ap_without = mean_metric(h_without, "future_purchase_ap")
    ablation_rows.append({
        "config": cfg, "feature_mode": params.feature_mode,
        "fit_population": "active_only" if params.active_only else "all_customers",
        "auc_with_cluster": auc_with, "auc_without_cluster": auc_without,
        "auc_gain": auc_with - auc_without if pd.notna(auc_with) and pd.notna(auc_without) else np.nan,
        "ap_with_cluster": ap_with, "ap_without_cluster": ap_without,
        "ap_gain": ap_with - ap_without if pd.notna(ap_with) and pd.notna(ap_without) else np.nan,
        "interpretation_note": "-1 encodes inactive status and duplicates information available from Frequency=0." if params.active_only else "Cluster representation ablation; exploratory only.",
    })
ablation_df = pd.DataFrame(ablation_rows)

summary_path = output_dir / "final_experiment_summary.csv"
summary_full_path = output_dir / "summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
final_summary.to_csv(summary_path, index=False)
final_summary.to_csv(summary_full_path, index=False)
holdout_all.to_csv(output_dir / "all_holdout_metrics.csv", index=False)
train_all.to_csv(output_dir / "all_train_metrics.csv", index=False)
ablation_df.to_csv(output_dir / "cluster_feature_ablation.csv", index=False)

# Output integrity manifest: only files actually present are listed.
output_manifest = []
for p in sorted(output_dir.rglob("*")):
    if p.is_file():
        output_manifest.append({"relative_path": str(p.relative_to(output_dir)), "bytes": p.stat().st_size})
(output_dir / "output_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print("PRIMARY SUMMARY")
display(final_summary.round(4))
print("\nCLUSTER FEATURE ABLATION — SECONDARY")
display(ablation_df.round(4))

# Excel workbook only includes outputs that this notebook actually created.
frames = {
    "Summary": final_summary,
    "Holdout_All": holdout_all,
    "Train_All": train_all,
    "Cluster_Ablation": ablation_df,
    "Summary_Block_CI": pd.read_csv(output_dir / "summary_block_ci.csv") if (output_dir / "summary_block_ci.csv").exists() else pd.DataFrame(),
    "Population": pd.read_csv(output_dir / "population_sensitivity.csv") if (output_dir / "population_sensitivity.csv").exists() else pd.DataFrame(),
    "Transitions": pd.read_csv(output_dir / "holdout_transitions.csv") if (output_dir / "holdout_transitions.csv").exists() else pd.DataFrame(),
    "Horizon": pd.read_csv(output_dir / "horizon_sensitivity.csv") if (output_dir / "horizon_sensitivity.csv").exists() else pd.DataFrame(),
    "Paired_Bootstrap": pd.read_csv(output_dir / "paired_block_bootstrap.csv") if (output_dir / "paired_block_bootstrap.csv").exists() else pd.DataFrame(),
    "Paired_Wilcoxon": pd.read_csv(output_dir / "paired_wilcoxon_holm_exploratory.csv") if (output_dir / "paired_wilcoxon_holm_exploratory.csv").exists() else pd.DataFrame(),
    "Cleaning_Audit": cleaning_audit,
}
xlsx_path = output_dir / "experiment_results_v61.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for sheet_name, frame in frames.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# Refresh manifest after workbook creation so it includes the actual workbook.
output_manifest = []
for p in sorted(output_dir.rglob("*")):
    if p.is_file() and p.name != "output_manifest.json":
        output_manifest.append({"relative_path": str(p.relative_to(output_dir)), "bytes": p.stat().st_size})
(output_dir / "output_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

print(f"\nWorkbook created: {xlsx_path}")
print("The output manifest lists only files actually produced by this run.")

## Cell 22 — Export outputs to Google Drive or ZIP

In [ ]:
from google.colab import drive, files

def export_outputs(source_dir, dest_name="adaptive_rolling_dynamic_rfm_v61"):
    source_dir = Path(source_dir)
    if not source_dir.exists():
        raise FileNotFoundError(f"Output directory does not exist: {source_dir}")
    try:
        drive.mount("/content/drive", force_remount=False)
        dest = Path("/content/drive/MyDrive") / dest_name
        shutil.copytree(source_dir, dest, dirs_exist_ok=True)
        print(f"Saved to Drive: {dest}")
        return dest
    except Exception as exc:
        print(f"Drive export unavailable: {exc}")

    zip_base = Path("/content") / dest_name
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", source_dir))
    print(f"ZIP: {zip_path} ({zip_path.stat().st_size:,} bytes)")
    try:
        files.download(str(zip_path))
    except Exception as exc:
        print(f"Direct download unavailable: {exc}")
    return zip_path


exported_path = export_outputs(output_dir)

## Cell 23 — Exact runtime versions and reproducibility summary

In [ ]:
package_names = ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "openpyxl", "pyarrow"]
resolved_versions = {}
for package in package_names:
    try:
        resolved_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        resolved_versions[package] = "not-installed"

req_path = output_dir / "requirements-resolved.txt"
req_lines = [f"{name}=={version}" for name, version in resolved_versions.items()
             if version != "not-installed"]
req_path.write_text("\n".join(req_lines) + "\n", encoding="utf-8")

REPRO["resolved_package_versions"] = resolved_versions
REPRO["output_files"] = [
    str(p.relative_to(output_dir)) for p in sorted(output_dir.rglob("*")) if p.is_file()
]
(output_dir / "reproducibility.json").write_text(
    json.dumps(REPRO, indent=2, default=str), encoding="utf-8"
)

print("Resolved package versions:")
print(json.dumps(resolved_versions, indent=2))
print("\nDataset SHA256:", REPRO.get("dataset_sha256"))
print("Cleaned rows:", REPRO.get("dataset_rows_cleaned"))
print("Customers:", REPRO.get("dataset_customers"))
print("Train cutoff:", REPRO.get("hyperparams_derived", {}).get("train_cutoff"))
print("Primary configs:", REPRO.get("comparison_groups", {}).get("primary_normalized_all_customer"))
print("No walk-forward claim is made by this notebook; the implemented evaluation is fixed-reference holdout + cross-sectional proxy tasks.")
print("Resolved requirements:", req_path)

# Refresh reproducibility and output manifests after requirements and workbook are present.
REPRO["resolved_package_versions"] = resolved_versions
REPRO["output_files"] = [
    str(p.relative_to(output_dir)) for p in sorted(output_dir.rglob("*"))
    if p.is_file() and p.name not in {"reproducibility.json", "output_manifest.json"}
]
(output_dir / "reproducibility.json").write_text(
    json.dumps(REPRO, indent=2, default=str), encoding="utf-8"
)
output_manifest = [
    {"relative_path": str(p.relative_to(output_dir)), "bytes": p.stat().st_size}
    for p in sorted(output_dir.rglob("*"))
    if p.is_file() and p.name != "output_manifest.json"
]
(output_dir / "output_manifest.json").write_text(
    json.dumps(output_manifest, indent=2), encoding="utf-8"
)

# Export only after every artifact has been generated.
exported_path = export_outputs(output_dir)

Cell 24 — Limitations & Threats to Validity (markdown)


## Limitations & Threats to Validity — v6.1

### Internal validity
1. **No verified event ground truth.** Future-purchase AUC/AP và operational-inactivity AUC/AP là proxy tasks; không phải Precision/Recall/F1 của event detection.
2. **Fixed-reference clustering.** Reference MiniBatchKMeans được fit một lần trên train và giữ nguyên trong holdout. Các stability metrics đo độ ổn định của assignment theo fixed centroids, không đo drift của centroids qua retraining.
3. **Adaptive cohort prior.** Cohort được xác định theo số ngày có giao dịch trong lookback. Đây là hierarchical cohort shrinkage heuristic, không phải mô hình chu kỳ mua đã được học/validated độc lập.
4. **Cross-sectional prediction.** Predictor được đánh giá bằng random stratified split khách hàng trong từng snapshot. Đây không phải temporal deployment test; kết quả phải được gọi là cross-sectional future-label discrimination.
5. **Two-stage inactive label.** Cluster `-1` tương đương trạng thái Frequency=0. Nó được báo cáo như trạng thái inactive, không được diễn giải là segment độc lập có thông tin mới.

### Comparability
6. **Active vs all population.** Silhouette/DBI/CH trên active customers và toàn bộ population trả lời hai câu hỏi khác nhau; không được xếp hạng chung.
7. **Raw vs rate features.** Raw RFM và normalized-rate RFM là ablation khác feature space; không được dùng như matched primary comparison.
8. **Matched fixed windows.** Fixed 98/112/182 ngày được bổ sung để kiểm tra liệu lợi ích của Adaptive đến từ cá nhân hóa hay chỉ do độ dài cửa sổ. Kết luận chỉ được đưa ra sau khi chạy đầy đủ và so trên cùng snapshots.

### Statistical validity
9. **Overlapping snapshots.** Snapshot cách nhau 14 ngày trong khi cửa sổ có thể đến 182 ngày, tạo phụ thuộc thời gian. Paired Wilcoxon được gắn nhãn exploratory; Holm chỉ xử lý multiple testing, không sửa temporal dependence.
10. **Block bootstrap support.** Bootstrap không circular và giữ các vị trí NaN. CI bị để trống khi không có đủ block liên tục; ngay cả khi có CI, 17 snapshots vẫn là chuỗi ngắn và CI chỉ mang tính exploratory.
11. **Churn coverage.** Churn horizon 90 ngày có ít snapshot hợp lệ hơn clustering metrics. Churn luôn được báo cáo riêng với số snapshot hợp lệ, không đưa vào ranking chính.
12. **Multiple metrics.** Quality, stability và predictive utility được báo cáo riêng. Không tính một composite rank-sum tùy ý.

### Data and reproducibility
13. **Cleaning scope.** Các giao dịch thiếu CustomerID, hóa đơn hủy và Quantity/Price không hợp lệ bị loại; cleaning audit ghi rõ số dòng ở từng bước. Kết quả chỉ đại diện cho population giao dịch còn lại.
14. **Cache integrity.** Parquet cache chỉ được tái sử dụng khi source SHA256 và cleaning-version khớp.
15. **Environment.** Notebook ghi nhận chính xác các package versions đã resolve và lưu requirements sau khi chạy; Colab base image/hardware vẫn có thể khác.

### What would strengthen the study further
- Synthetic drift scenarios với event log ground truth để đo Precision, Recall, F1, false-alert rate và detection delay.
- Walk-forward/rolling-origin prediction với model supervised được fit chỉ trên giai đoạn trước.
- Dynamic reclustering tại từng snapshot và cluster matching một-một nếu mục tiêu là theo dõi centroid evolution.
- External validation trên dataset retail khác.
- Prespecified equivalence margins/TOST nếu muốn kết luận Adaptive tương đương Fixed.